## Cost Estimation of Queries

In [ ]:
from google.cloud import bigquery

def estimate_query_cost(query: str) -> float:
    """
    Estimate the cost of a BigQuery query using a dry run.

    Parameters:
      query (str): The SQL query to estimate.
      cost_per_tb (float): Cost per terabyte processed in your billing currency (default is 5.0).

    Returns:
      float: The estimated cost to run the query.
    """
    # Initialize the BigQuery client
    client = bigquery.Client()

    # Configure the query job for a dry run
    job_config = bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False
    )

    cost_per_tib = 6.25

    # Submit the query as a dry run
    query_job = client.query(query, job_config=job_config)

    # Retrieve the total number of bytes that would be processed
    bytes_processed = query_job.total_bytes_processed
    print(f"Total bytes processed (dry run): {bytes_processed}")

    # Calculate the estimated cost (convert bytes to terabytes)
    estimated_cost = (bytes_processed / 2**40) * cost_per_tib
    print(f"Estimated cost to run the query in dollars: {estimated_cost}")

    return estimated_cost




### Dataform

In [2]:
staging_table_dataform = """WITH
  source_data AS (
  SELECT
    event_name AS event_name,
  event_date AS event_date,
  event_timestamp AS event_timestamp,
  user_pseudo_id AS user_pseudo_id,
  (SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_id' LIMIT 1) AS ga_session_id,
  ecommerce.purchase_revenue AS purchase_revenue,
  ecommerce.total_item_quantity AS total_quantity,
  device.category AS device_category,
  (SELECT value.string_value FROM UNNEST(event_params) WHERE key = 'page_location' LIMIT 1) AS page_location
  FROM
    `non_copy_analytics_305832606.events_20241109`
  ),
  filtered_data AS (
  SELECT
    *
  FROM
    source_data
  WHERE event_name IN ('view_item', 'select_item', 'add_to_cart', 'begin_checkout', 'purchase')  )
SELECT
  PARSE_DATE('%Y%m%d', CAST(event_date AS STRING)) AS event_date,
  event_timestamp,
  event_name,
  user_pseudo_id,
  ga_session_id,
  purchase_revenue,
  total_quantity,
  device_category,
  page_location
FROM
  filtered_data
WHERE
  (user_pseudo_id IS NOT NULL
    OR ga_session_id IS NOT NULL)
"""

stg_1_day_run = estimate_query_cost (staging_table_dataform)

Total bytes processed (dry run): 4625863122
Estimated cost to run the query in dollars: 0.0289116445125


In [ ]:
open_user_funnel = """SELECT
  event_date AS event_date,
  event_name AS event_name,
  COUNT(DISTINCT user_pseudo_id) AS event_count
FROM `my-project-se.dataform_funnels_c_test_Arina.staged_data`
WHERE event_date = PARSE_DATE('%Y%m%d', '20241109')
GROUP BY
  event_date,
  event_name
 """

open_funnel_1_day_run = estimate_query_cost(open_user_funnel)

Total bytes processed (dry run): 21552804
Estimated cost to run the query in dollars: 0.000134705025


In [ ]:
close_user_funnel = """WITH
  source_data AS (
  SELECT
    *
  FROM
    `my-project-se.dataform_funnels_c_test_Arina.staged_data`
  WHERE event_date = PARSE_DATE('%Y%m%d', '20241109')
    ),

cte_step1 AS (
SELECT

event_date AS event_date
,
user_pseudo_id AS step1_id,
event_timestamp AS step1_timestamp
FROM source_data
WHERE event_name = 'view_item'
),

cte_step2 AS (
SELECT

event_date AS event_date
,
user_pseudo_id AS step2_id,
event_timestamp AS step2_timestamp
FROM source_data
WHERE event_name = 'select_item'
),

cte_step3 AS (
SELECT

event_date AS event_date
,
user_pseudo_id AS step3_id,
event_timestamp AS step3_timestamp
FROM source_data
WHERE event_name = 'add_to_cart'
),

cte_step4 AS (
SELECT

event_date AS event_date
,
user_pseudo_id AS step4_id,
event_timestamp AS step4_timestamp
FROM source_data
WHERE event_name = 'begin_checkout'
),

cte_step5 AS (
SELECT

event_date AS event_date
,
user_pseudo_id AS step5_id,
event_timestamp AS step5_timestamp,
purchase_revenue,
total_quantity
FROM source_data
WHERE event_name = 'purchase'
)
SELECT
  'view_item' AS event_name,
  cte_step1.event_date
  ,
  COUNT(DISTINCT cte_step1.step1_id) AS event_count,

  0 AS total_revenue,
  0 AS total_item_quantity
FROM cte_step1

GROUP BY
  cte_step1.event_date

UNION ALL

SELECT
  'select_item' AS event_name,
  cte_step1.event_date
  ,
  COUNT(DISTINCT cte_step2.step2_id) AS event_count,

  0 AS total_revenue,
  0 AS total_item_quantity
FROM cte_step1

LEFT JOIN cte_step2 ON cte_step1.event_date = cte_step2.event_date
  AND cte_step1.step1_id = cte_step2.step2_id
  AND cte_step1.step1_timestamp < cte_step2.step2_timestamp
GROUP BY
  cte_step1.event_date

UNION ALL

SELECT
  'add_to_cart' AS event_name,
  cte_step1.event_date
  ,
  COUNT(DISTINCT cte_step3.step3_id) AS event_count,

  0 AS total_revenue,
  0 AS total_item_quantity
FROM cte_step1

LEFT JOIN cte_step2 ON cte_step1.event_date = cte_step2.event_date
  AND cte_step1.step1_id = cte_step2.step2_id
  AND cte_step1.step1_timestamp < cte_step2.step2_timestamp
LEFT JOIN cte_step3 ON cte_step1.event_date = cte_step3.event_date
  AND cte_step2.step2_id = cte_step3.step3_id
  AND cte_step2.step2_timestamp < cte_step3.step3_timestamp
GROUP BY
  cte_step1.event_date

UNION ALL

SELECT
  'begin_checkout' AS event_name,
  cte_step1.event_date
  ,
  COUNT(DISTINCT cte_step4.step4_id) AS event_count,

  0 AS total_revenue,
  0 AS total_item_quantity
FROM cte_step1

LEFT JOIN cte_step2 ON cte_step1.event_date = cte_step2.event_date
  AND cte_step1.step1_id = cte_step2.step2_id
  AND cte_step1.step1_timestamp < cte_step2.step2_timestamp
LEFT JOIN cte_step3 ON cte_step1.event_date = cte_step3.event_date
  AND cte_step2.step2_id = cte_step3.step3_id
  AND cte_step2.step2_timestamp < cte_step3.step3_timestamp
LEFT JOIN cte_step4 ON cte_step1.event_date = cte_step4.event_date
  AND cte_step3.step3_id = cte_step4.step4_id
  AND cte_step3.step3_timestamp < cte_step4.step4_timestamp
GROUP BY
  cte_step1.event_date

UNION ALL

SELECT
  'purchase' AS event_name,
  cte_step1.event_date
  ,
  COUNT(DISTINCT cte_step5.step5_id) AS event_count,

  SUM(cte_step5.purchase_revenue) AS total_revenue,
  SUM(cte_step5.total_quantity) AS total_item_quantity
FROM cte_step1

LEFT JOIN cte_step2 ON cte_step1.event_date = cte_step2.event_date
  AND cte_step1.step1_id = cte_step2.step2_id
  AND cte_step1.step1_timestamp < cte_step2.step2_timestamp
LEFT JOIN cte_step3 ON cte_step1.event_date = cte_step3.event_date
  AND cte_step2.step2_id = cte_step3.step3_id
  AND cte_step2.step2_timestamp < cte_step3.step3_timestamp
LEFT JOIN cte_step4 ON cte_step1.event_date = cte_step4.event_date
  AND cte_step3.step3_id = cte_step4.step4_id
  AND cte_step3.step3_timestamp < cte_step4.step4_timestamp
LEFT JOIN cte_step5 ON cte_step1.event_date = cte_step5.event_date
  AND cte_step4.step4_id = cte_step5.step5_id
  AND cte_step4.step4_timestamp < cte_step5.step5_timestamp
GROUP BY
  cte_step1.event_date"""

close_funnel_1_day_run = estimate_query_cost(close_user_funnel)

Total bytes processed (dry run): 26081916
Estimated cost to run the query in dollars: 0.000163011975


In [5]:
total_costs_for_1_year = (stg_1_day_run * 2 + open_funnel_1_day_run * 4  + close_funnel_1_day_run * 4) * 365

print(total_costs_for_1_year)

21.560876684125


### Scheduled Queries

In [ ]:
sch_query_close_funnel = """ WITH
  source_data AS (
  SELECT
    event_name AS event_name,
  event_date AS event_date,
  event_timestamp AS event_timestamp,
  user_pseudo_id AS user_pseudo_id,
  (SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_id' LIMIT 1) AS ga_session_id,
  ecommerce.purchase_revenue AS purchase_revenue,
  ecommerce.total_item_quantity AS total_quantity,
  device.category AS device_category,
  (SELECT value.string_value FROM UNNEST(event_params) WHERE key = 'page_location' LIMIT 1) AS page_location
  FROM
    `non_copy_analytics_305832606.events_20241109`
  ),

cte_step1 AS (
SELECT

event_date AS event_date,
device_category
,
user_pseudo_id AS step1_id,
event_timestamp AS step1_timestamp
FROM source_data
WHERE event_name = 'view_item'
),

cte_step2 AS (
SELECT

event_date AS event_date,
device_category
,
user_pseudo_id AS step2_id,
event_timestamp AS step2_timestamp
FROM source_data
WHERE event_name = 'select_item'
),

cte_step3 AS (
SELECT

event_date AS event_date,
device_category
,
user_pseudo_id AS step3_id,
event_timestamp AS step3_timestamp
FROM source_data
WHERE event_name = 'add_to_cart'
),

cte_step4 AS (
SELECT

event_date AS event_date,
device_category
,
user_pseudo_id AS step4_id,
event_timestamp AS step4_timestamp
FROM source_data
WHERE event_name = 'begin_checkout'
),

cte_step5 AS (
SELECT

event_date AS event_date,
device_category
,
user_pseudo_id AS step5_id,
event_timestamp AS step5_timestamp,
purchase_revenue,
total_quantity
FROM source_data
WHERE event_name = 'purchase'
)
SELECT
  'view_item' AS event_name,
  cte_step1.event_date,
  cte_step1.device_category
  ,
  COUNT(DISTINCT cte_step1.step1_id) AS event_count,

  0 AS total_revenue,
  0 AS total_item_quantity
FROM cte_step1

GROUP BY
  cte_step1.event_date
  , cte_step1.device_category

UNION ALL

SELECT
  'select_item' AS event_name,
  cte_step1.event_date,
  cte_step1.device_category
  ,
  COUNT(DISTINCT cte_step2.step2_id) AS event_count,

  0 AS total_revenue,
  0 AS total_item_quantity
FROM cte_step1

LEFT JOIN cte_step2 ON cte_step1.event_date = cte_step2.event_date
  AND cte_step1.device_category = cte_step2.device_category
  AND cte_step1.step1_id = cte_step2.step2_id
  AND cte_step1.step1_timestamp < cte_step2.step2_timestamp
GROUP BY
  cte_step1.event_date
  , cte_step1.device_category

UNION ALL

SELECT
  'add_to_cart' AS event_name,
  cte_step1.event_date,
  cte_step1.device_category
  ,
  COUNT(DISTINCT cte_step3.step3_id) AS event_count,

  0 AS total_revenue,
  0 AS total_item_quantity
FROM cte_step1

LEFT JOIN cte_step2 ON cte_step1.event_date = cte_step2.event_date
  AND cte_step1.device_category = cte_step2.device_category
  AND cte_step1.step1_id = cte_step2.step2_id
  AND cte_step1.step1_timestamp < cte_step2.step2_timestamp
LEFT JOIN cte_step3 ON cte_step1.event_date = cte_step3.event_date
  AND cte_step1.device_category = cte_step3.device_category
  AND cte_step2.step2_id = cte_step3.step3_id
  AND cte_step2.step2_timestamp < cte_step3.step3_timestamp
GROUP BY
  cte_step1.event_date
  , cte_step1.device_category

UNION ALL

SELECT
  'begin_checkout' AS event_name,
  cte_step1.event_date,
  cte_step1.device_category
  ,
  COUNT(DISTINCT cte_step4.step4_id) AS event_count,

  0 AS total_revenue,
  0 AS total_item_quantity
FROM cte_step1

LEFT JOIN cte_step2 ON cte_step1.event_date = cte_step2.event_date
  AND cte_step1.device_category = cte_step2.device_category
  AND cte_step1.step1_id = cte_step2.step2_id
  AND cte_step1.step1_timestamp < cte_step2.step2_timestamp
LEFT JOIN cte_step3 ON cte_step1.event_date = cte_step3.event_date
  AND cte_step1.device_category = cte_step3.device_category
  AND cte_step2.step2_id = cte_step3.step3_id
  AND cte_step2.step2_timestamp < cte_step3.step3_timestamp
LEFT JOIN cte_step4 ON cte_step1.event_date = cte_step4.event_date
  AND cte_step1.device_category = cte_step4.device_category
  AND cte_step3.step3_id = cte_step4.step4_id
  AND cte_step3.step3_timestamp < cte_step4.step4_timestamp
GROUP BY
  cte_step1.event_date
  , cte_step1.device_category

UNION ALL

SELECT
  'purchase' AS event_name,
  cte_step1.event_date,
  cte_step1.device_category
  ,
  COUNT(DISTINCT cte_step5.step5_id) AS event_count,

  SUM(cte_step5.purchase_revenue) AS total_revenue,
  SUM(cte_step5.total_quantity) AS total_item_quantity
FROM cte_step1

LEFT JOIN cte_step2 ON cte_step1.event_date = cte_step2.event_date
  AND cte_step1.device_category = cte_step2.device_category
  AND cte_step1.step1_id = cte_step2.step2_id
  AND cte_step1.step1_timestamp < cte_step2.step2_timestamp
LEFT JOIN cte_step3 ON cte_step1.event_date = cte_step3.event_date
  AND cte_step1.device_category = cte_step3.device_category
  AND cte_step2.step2_id = cte_step3.step3_id
  AND cte_step2.step2_timestamp < cte_step3.step3_timestamp
LEFT JOIN cte_step4 ON cte_step1.event_date = cte_step4.event_date
  AND cte_step1.device_category = cte_step4.device_category
  AND cte_step3.step3_id = cte_step4.step4_id
  AND cte_step3.step3_timestamp < cte_step4.step4_timestamp
LEFT JOIN cte_step5 ON cte_step1.event_date = cte_step5.event_date
  AND cte_step1.device_category = cte_step5.device_category
  AND cte_step4.step4_id = cte_step5.step5_id
  AND cte_step4.step4_timestamp < cte_step5.step5_timestamp
GROUP BY
  cte_step1.event_date
  , cte_step1.device_category """

sch_query_close_1_day = estimate_query_cost(sch_query_close_funnel)

Total bytes processed (dry run): 40747217492
Estimated cost to run the query in dollars: 0.254670109325


In [7]:
sch_query_open_funnel = """ SELECT
  event_date AS event_date,
  event_name AS event_name,
  COUNT(DISTINCT user_pseudo_id) AS event_count
FROM `non_copy_analytics_305832606.events_20241109`
WHERE event_name IN ('view_item', 'select_item', 'add_to_cart', 'begin_checkout', 'purchase')
GROUP BY
  event_date,
  event_name
 """

sch_query_open_1_day = estimate_query_cost (sch_query_open_funnel)

Total bytes processed (dry run): 436338770
Estimated cost to run the query in dollars: 0.0027271173125


#### Without staging table

In [10]:
total_costs_sch_1_year = (2 * (sch_query_open_1_day) + 2 * (sch_query_close_1_day))*365*365
print(total_costs_sch_1_year)

68583.49103756188


With staging table

In [12]:
total_costs_sch_stg_1_year = (stg_1_day_run  + open_funnel_1_day_run * 2  + close_funnel_1_day_run * 2) * 365 * 365
print(total_costs_sch_stg_1_year)

3931.0805348278127
